# Tune all

Run GridSearchCV + BER threshold tuning for **all four** benchmark models.
Writes `data/processed/tuned/<model_id>.json` per model.

Per-model notebooks: `linear_lr.ipynb`, `topk_rf.ipynb`, `topk_knn.ipynb`, `topk_xgb.ipynb`.

**Stage 1:** hyperparameters — max mean PR AUC.

**Stage 2:** classifier threshold — minimize mean BER.

Shared sensor branch adds an **isolation forest** `decision_function` score after T² and hub interactions (unsupervised, refit per CV fold). Grid includes `isolation_forest__n_estimators`.

After hubs: **neighbor_fail_rate** (RF-weighted kNN mean train-label rate; tune `neighbor_fail_rate__n_neighbors`), then isolation forest.


In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_IDS = list(MODEL_SPECS)
# MODEL_IDS = ["linear_lr"]  # uncomment to tune a subset


In [ ]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


In [ ]:
for MODEL_ID in MODEL_IDS:
    spec = MODEL_SPECS[MODEL_ID]
    param_grid = spec.make_param_grid()
    print(MODEL_ID)
    display(pd.DataFrame([{k: v} for k, v in param_grid.items()]))


In [ ]:
searches = {}
for MODEL_ID in MODEL_IDS:
    spec = MODEL_SPECS[MODEL_ID]
    search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
    print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
    searches[MODEL_ID] = fit_with_progress(search, X_train, y_train)


In [ ]:
summaries = {}
for MODEL_ID in MODEL_IDS:
    spec = MODEL_SPECS[MODEL_ID]
    cv_summary, fold_results, aggregated = summarize_cv_search(searches[MODEL_ID], spec)
    summaries[MODEL_ID] = (cv_summary, fold_results, aggregated)
    print(f"\n=== {MODEL_ID} Stage 1 best (mean PR AUC) ===")
    display(aggregated.head(10))


In [ ]:
thresholds = {}
for MODEL_ID in MODEL_IDS:
    spec = MODEL_SPECS[MODEL_ID]
    cv_summary, fold_results, aggregated = summaries[MODEL_ID]
    threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
    thresholds[MODEL_ID] = threshold_result
    print(f"\n=== {MODEL_ID} Stage 2 ===")
    print(f"  best threshold: {threshold_result['best_threshold']:.2f}")
    print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
    display(threshold_result["per_threshold_mean_ber"].head(10))


In [ ]:
payloads = {}
for MODEL_ID in MODEL_IDS:
    spec = MODEL_SPECS[MODEL_ID]
    cv_summary, fold_results, aggregated = summaries[MODEL_ID]
    threshold_result = thresholds[MODEL_ID]
    payloads[MODEL_ID] = save_tuned_params(
        spec,
        cv_summary,
        fold_results,
        aggregated,
        threshold_result=threshold_result,
    )
    print(f"Wrote {tuned_params_path(MODEL_ID)}")

{mid: payloads[mid]["grid_search_best_params"] for mid in MODEL_IDS}
